# Linear-equivalent disc / annulus with cone linearization
**The experiment.** Every natural-image patch is shown three ways, interleaved:

| `stimulusTag` | what it is |
|---|---|
| `image` | the image patch itself |
| `intensity` | uniform disc at the linear-equivalent intensity (patch averaged over the RF) |
| `linConeIntensity` / `lin cone intensity` | uniform disc at the **cone-linearized** equivalent intensity — averaged after a Weber cone nonlinearity `I / (I + WeberConstant)` |

Comparing image against each disc asks how much of the cell's preference for the real image
survives when the averaging happens in cone-response space instead of intensity space. The
measure is the nonlinearity index per patch, exactly as `computeNLI` defines it:

$$\mathrm{NLI} = \frac{\mathrm{image} - \mathrm{disc}}{|\mathrm{image}| + |\mathrm{disc}|}$$

set to zero when neither response clears a recording-mode threshold

In [ ]:
import contextlib
import io

import retinanalysis as ra
import numpy as np
import pandas as pd

from retinanalysis.SCutils import explore as sc
from retinanalysis.SCutils.protocols import linear_equivalent_disc as led

## 1. Find experiment dates and cells for one protocol

Set `PROC` to one of the three exact protocol names below. The compact table has one row per
experiment, cell, and short protocol. `date_index` numbers each experiment date so it can
be selected directly in Section 1a. The table also includes `cell_type_short`, the epoch-group
`recording_technique`, the resolved `onlineAnalysis`, and all numeric FilterWheel values used
for that cell. The matching blocks are loaded and their recording mode is quietly resolved
from the amplifier, so missing or incorrect `onlineAnalysis` labels do not pass downstream.
Response paths are fetched in one batch, and a quick representative
series-resistance read excludes blocks above the cutoff; the full per-recording analysis
checks the raw data again. The original label is retained in `onlineAnalysis_recorded`. For
`LinearEquivalentDisc`, blocks
without the `linearizeCones` option are the older unrelated experiment and are also excluded.

In [ ]:
# Choose exactly one:
# 'LinearEquivalentAnnulus', 'LinearEquivalentDiscConeLin', or 'LinearEquivalentDisc'
PROC = 'LinearEquivalentAnnulus'

# One database discovery supplies both the compact overview and downstream blocks.
df_blocks = led.find_blocks(protocols=[PROC], show=False)
# Resolve the recording mode in the background.
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    df_blocks = led.check_series_resistance(
        df_blocks, show=False, sample_series_resistance=True)

protocol_cells = led.protocol_cells_from_blocks(df_blocks)

### 1a. Choose a date and group its recordings

Choose one `date_index` from the Section 1 table. Only that date is grouped below, so the
overview is not repeated for every experiment. Each row is one cell × resolved recording
mode × disc site × light setting, with blocks from the same condition pooled. The tree table
keeps only: `exp_name`, `cell_label`, `cell_type_short`,
resolved `onlineAnalysis`, `site`, `light_settings`, `block_ids`, `image_names`, short
`protocols`, and protocol-setting `maxIntensity`. A blank `maxIntensity` means the older
protocol did not record that setting.

In [ ]:
# Choose a date_index shown in Section 1.
DATE_INDEX = 36

date_rows = protocol_cells.loc[protocol_cells.date_index.eq(DATE_INDEX)]
if date_rows.empty:
    raise ValueError(f'date_index {DATE_INDEX} is not available for {PROC}')
EXP_NAME = date_rows.exp_name.iloc[0]

selected_blocks = df_blocks.loc[df_blocks.exp_name.eq(EXP_NAME)].copy()
if selected_blocks.empty:
    raise ValueError(f'{EXP_NAME!r} is not available for {PROC}')
groups = led.group_blocks(selected_blocks)

## 2. What the cell was shown

Continue from the date selected in Section 1a. The dropdown contains every recorded `imageName`
for that date; choosing one redraws an example patch and its two equivalent discs. The three
stimuli use a common grey scale: the natural-image patch as seen through
the aperture, the uniform disc at the linear-equivalent intensity, and the uniform disc at the
cone-linearized intensity. The image is loaded from the van Hateren resources in the turner
package the same way `NaturalImageFlashProtocol.m` does — big-endian `.iml`, rescaled so the
brightest pixel is 1, at 6.6 µm per image pixel.

A useful check that the geometry is right: the mean intensity inside the aperture should land on
the recorded `equivalentIntensity`, since that is what the equivalent disc is defined to be.

In [ ]:
image_example = led.stimulus_example_widget(selected_blocks)
image_example

## 3. Analyze one recording

Per epoch the onset and offset responses are measured, then averaged within
(image, patch, stimulus category). A patch is kept only if it has an image trial and at least
one disc trial. Left: each patch's image response against its two discs, with the unity line —
points above it are patches where the real image drove the cell more than the equivalent
uniform disc. Right: the NLI values those produce.

In [ ]:
row = groups.sort_values('epochs', ascending=False).iloc[2]

# Before analyzing a single recording, see what else this cell has: the other
# conditions it was recorded in, and whether the one picked is representative.
led.describe_cell(f'{row.exp_name}/{row.cell_label}', groups)

rec = led.analyze_group(row.exp_name, [int(b) for b in row.block_ids.split(',')],
                        online_analysis=row.onlineAnalysis)
led.plot_group(rec);

## 4. Batch analyze and save

Records go to `<OUTPUT_DIR>/linear_equivalent_disc/records.h5` with a `summary.csv` index,
upserted per (experiment, cell, mode, site, filter wheel, background) — same layout as the other
protocols, in its own directory. `skip_existing` makes re-running the notebook cheap.

In [ ]:
records = led.analyze_all(groups, save=True, plot=False, skip_existing=True)

## 5. Population: does cone linearization remove the nonlinearity?

Section 1 of `populationLinConeDisc.m`: one line per recording joining its mean standard-disc
NLI to its mean cone-linearized NLI, per recording mode, with a paired Wilcoxon signed-rank
test. If cone linearization captures what the cell is doing, the cone-linearized NLI should sit
closer to zero.

In [ ]:
summary = led.load_summary()
print(f'{len(summary)} stored recordings')
led.plot_population_nli(summary, window='onset');

In [ ]:
led.plot_population_nli(summary, window='offset');

### Pooled per-patch distributions

Section 2 of the population script — every patch from every recording, no per-cell averaging,
with a two-sample KS test.

In [ ]:
led.plot_nli_distributions(window='onset');

### Split by disc site

The annulus protocol puts the disc over the surround; the other two put it over the centre.
Those are different experiments and are worth looking at separately.

In [ ]:
for site in sorted(summary['site'].unique()):
    sub = summary[summary['site'].eq(site)]
    print(f'--- disc over {site}: {len(sub)} recordings ---')
    display(sub.groupby('online_analysis')[['nli_disc_onset', 'nli_cone_onset',
                                            'nli_disc_offset', 'nli_cone_offset']]
               .agg(['count', 'mean']).round(3))

## 6. Come back later

`load_summary()` reads the scalar index with no DataJoint or SSD access; `load_records()` pulls
the per-patch arrays.

In [ ]:
cols = ['exp_name', 'cell_label', 'cell_type', 'online_analysis', 'online_analysis_recorded',
        'site', 'light_setting', 'max_intensity', 'series_resistance', 'n_patches',
        'nli_disc_onset', 'nli_cone_onset']
sc.scroll_table(summary[[c for c in cols if c in summary.columns]], height=320)

## Inspect one cell

Everything above runs across the whole dataset. This section goes the other way: name a single
cell and see every recording of it, split by condition.

A cell is identified by `'<experiment>/<cell label>'`, e.g. `'2026-04-03_E/Cell4'`. `list_cells` shows the
available ids together with the conditions each cell was recorded in — recording mode, site,
light setting, and whatever else varies for this protocol — so you can pick one. A bare cell
label also works when it is unambiguous.

In [ ]:
led.list_cells(groups)

In [ ]:
# Change CELL to inspect a different one. Each condition is analyzed and
# plotted in turn, and the records are returned for further poking.
CELL = '2026-04-03_E/Cell4'

cell_records = led.inspect_cell(CELL, groups)